# The prompts

Five ways of asking one question. Every method ends with the same instruction
about the answer format, so that a difference between methods is a difference in
how the question was asked rather than in how the answer was requested.

In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd

In [ ]:
if Path.cwd().name == 'notebooks':
    %cd ..

sys.path.insert(0, str(Path('scripts').resolve()))

In [ ]:
%load_ext autoreload
%autoreload 2

import backends
import evaluate
import prompts as templates
import run
import settings
import utils

utils.make_directories()
pd.set_option('display.max_colwidth', 90)
# Built from the dataset rather than shipped, so that what the pipeline reads is
# always derived from what is in data/halomi rather than from a stale copy.
if not settings.ITEMS_PATH.exists():
    raise SystemExit(
        'Nothing has been built yet. From the repository root, run:\n'
        '    python scripts/build.py\n'
        'That writes data/benchmark/items.csv and prompts.csv, which every '
        'notebook and stage reads.')

print('Ready')

## What was built

In [ ]:
prompts = utils.read_table(settings.PROMPTS_PATH)
print(f'{len(prompts):,} prompts')
display(prompts.groupby(['method', 'shots']).size().to_frame('prompts'))

## One of each, in full

In [ ]:
for method in sorted(prompts['method'].unique()):
    row = prompts[prompts['method'] == method].iloc[0]
    print('=' * 78)
    print(f"{method}, {row['shots']} examples, expecting {row['answer']}")
    print('=' * 78)
    print(row['prompt'])
    print()

## Length

Chain of translation and span level ask for reasoning before the label, so their
prompts and their replies both run longer. The token cap is set per method for
that reason, and a cap that suits the baseline would truncate the others.

In [ ]:
lengths = prompts.assign(words=prompts['prompt'].str.split().str.len())
display(lengths.groupby('method')['words'].describe()[['mean', '50%', 'max']]
        .round(0))
print()
for method, spec in settings.METHODS.items():
    print(f"  {method:<12} cap {spec['max_tokens']:>4} tokens")